In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch

In [2]:
def generate_time_series(batch_size, n_steps):
    freq1, freq2, offsets1, offsets2 = np.random.rand(4, batch_size, 1)
    time = np.linspace(0, 1, n_steps)
    series = 0.5 * np.sin((time - offsets1) * (freq1 * 10 + 10)) # Onda 1
    series += 0.2 * np.sin((time - offsets2) * (freq2 * 20 + 20)) # Onda 2
    series += 0.1 * (np.random.rand(batch_size, n_steps) - 0.5) # Ruido
    return series[..., np.newaxis].astype(np.float32)

In [3]:
batch_size = 10_000
n_steps = 50

series = generate_time_series(batch_size, n_steps+1)
X_train, y_train = series[:7000, :n_steps] , series[:7000, -1]
X_valid, y_valid = series[7000:9000, :n_steps], series[7000:9000, -1]
X_test, y_test = series[9000:,:n_steps], series[9000:, :-1]

X_train , y_train = torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float()
X_test , y_test = torch.from_numpy(X_test).float(), torch.from_numpy(y_test).float()
X_valid, y_valid = torch.from_numpy(X_valid).float(), torch.from_numpy(y_valid).float()

In [4]:
import torch
from torch.nn import functional as F
from torch import nn

In [5]:
class SimpleRNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.simple_rnn = nn.RNN(50, 1)

  def forward(self, x):
    print("Input:", x.shape)
    x = x.flatten(1)
    print("Flatten:", x.shape)
    out, _ = self.simple_rnn(x)
    print("Output:", out.shape)
    return out

In [6]:
model = SimpleRNN()

out = model(X_train)

Input: torch.Size([7000, 50, 1])
Flatten: torch.Size([7000, 50])
Output: torch.Size([7000, 1])


In [7]:
class DeepRNN(nn.Module):
    def __init__(self, output_size):
        super().__init__()
        self.simple_rnn = nn.RNN(1, 20, batch_first=True)
        self.simple_rnn2 = nn.RNN(20, 20, batch_first=True)
        self.output = nn.RNN(20, output_size, batch_first=True)

    def forward(self, x):
        print("Input:", x.shape)
        x, _ = self.simple_rnn(x)
        x, _ = self.simple_rnn2(x)
        x, _ = self.output(x)
        print("Output:", x.shape)
        return x

In [8]:
model = DeepRNN(output_size=1)

n_steps = 50
series = generate_time_series(1, n_steps + 10)
X_new, y_new = series[:, :n_steps], series[:, n_steps:]
X_new, y_new = torch.from_numpy(X_new).float(), torch.from_numpy(y_new).float()
X = X_new

y_pred = []

for step_ahead in range(10):
    out = model(X[:, step_ahead:])
    y_pred.append(out[:, -1:, :])

y_pred = torch.cat(y_pred, dim=1)
print("Pred shape:", y_pred.shape)

Input: torch.Size([1, 50, 1])
Output: torch.Size([1, 50, 1])
Input: torch.Size([1, 49, 1])
Output: torch.Size([1, 49, 1])
Input: torch.Size([1, 48, 1])
Output: torch.Size([1, 48, 1])
Input: torch.Size([1, 47, 1])
Output: torch.Size([1, 47, 1])
Input: torch.Size([1, 46, 1])
Output: torch.Size([1, 46, 1])
Input: torch.Size([1, 45, 1])
Output: torch.Size([1, 45, 1])
Input: torch.Size([1, 44, 1])
Output: torch.Size([1, 44, 1])
Input: torch.Size([1, 43, 1])
Output: torch.Size([1, 43, 1])
Input: torch.Size([1, 42, 1])
Output: torch.Size([1, 42, 1])
Input: torch.Size([1, 41, 1])
Output: torch.Size([1, 41, 1])
Pred shape: torch.Size([1, 10, 1])


In [9]:
model = DeepRNN(10)
series = generate_time_series(10_000, n_steps+10)

X_train, y_train = series[:7_000,:n_steps], series[7_000:, -10:, 0]
X_valid, y_valid = series[7_000:9_000], series[7_000:9_000, -10:, 0]
X_test, y_test = series[9_000:,:n_steps], series[9000:, -10, 0]

X_train, y_train = torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float
X_valid, y_valid = torch.from_numpy(X_valid).float(), torch.from_numpy(y_valid).float
X_test, y_test = torch.from_numpy(X_test).float(), torch.from_numpy(y_test).float

y_pred = []

for step_ahead in range(10):
    out = model(X_train[:, step_ahead:])
    y_pred.append(out[:, -10:, :])

y_pred = torch.cat(y_pred, dim=1)

Input: torch.Size([7000, 50, 1])
Output: torch.Size([7000, 50, 10])
Input: torch.Size([7000, 49, 1])
Output: torch.Size([7000, 49, 10])
Input: torch.Size([7000, 48, 1])
Output: torch.Size([7000, 48, 10])
Input: torch.Size([7000, 47, 1])
Output: torch.Size([7000, 47, 10])
Input: torch.Size([7000, 46, 1])
Output: torch.Size([7000, 46, 10])
Input: torch.Size([7000, 45, 1])
Output: torch.Size([7000, 45, 10])
Input: torch.Size([7000, 44, 1])
Output: torch.Size([7000, 44, 10])
Input: torch.Size([7000, 43, 1])
Output: torch.Size([7000, 43, 10])
Input: torch.Size([7000, 42, 1])
Output: torch.Size([7000, 42, 10])
Input: torch.Size([7000, 41, 1])
Output: torch.Size([7000, 41, 10])


In [10]:
class LSTMModel(nn.Module):
    def __init__(self, output_size):
        super().__init__()
        self.lstm = nn.LSTM(1, 20, batch_first=True)
        self.output = nn.LSTM(20, output_size, batch_first=True)

    def forward(self, x):
        print("Input:", x.shape)
        x1, _ = self.lstm(x)
        print("LSTM:", x1.shape)
        out, _ = self.output(x1)
        print("Output:", out.shape)
        return out

In [11]:
model_lstm = LSTMModel(10)

model_lstm(X_train).shape

Input: torch.Size([7000, 50, 1])
LSTM: torch.Size([7000, 50, 20])
Output: torch.Size([7000, 50, 10])


torch.Size([7000, 50, 10])

In [12]:
class ModelGRU(nn.Module):
  def __init__(self):
    super().__init__()
    self.gru = nn.GRU(1, 20)
    self.output = nn.GRU(20, 10)

  def forward(self, x):
    print("Input:", x.shape)
    x, _ = self.gru(x)
    print("Input:", x.shape)
    out, _ = self.output(x)
    print("Output:", out.shape)
    return out

In [13]:
model_gru = ModelGRU()

model_gru(X_train).shape

Input: torch.Size([7000, 50, 1])
Input: torch.Size([7000, 50, 20])
Output: torch.Size([7000, 50, 10])


torch.Size([7000, 50, 10])